In [ ]:
!pip install geopandas osmnx shapely fiona

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.1 MB/s eta 0:00:00


# MEJORAMIENTO DE DATOS

In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
import random
from datetime import datetime
import calendar
import os
from google.colab import drive

# ==============================================================================
# CONFIGURACIÓN Y MONTAJE
# ==============================================================================
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN/'
INPUT_FILE = os.path.join(BASE_DIR, 'Datos_Crudos', 'dataset_integrador.csv')
OUTPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'dataset_gnn_granular_final.csv')



Mounted at /content/drive


# PASO 1: Instalacion de dependencias

In [4]:
# ==============================================================================
# CELDA 1: INSTALACIÓN E IMPORTACIONES
# ==============================================================================
!pip install geopandas osmnx shapely fiona pyarrow

import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
import random
from datetime import datetime
import calendar
import os
from google.colab import drive

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


# PASO 2: Configuración y Conexión con Google Drive

In [5]:
# ==============================================================================
# CELDA 2: CONFIGURACIÓN Y RUTAS
# ==============================================================================
# 1. Montar el disco
drive.mount('/content/drive')

# 2. Definir rutas
BASE_DIR = '/content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN'
INPUT_FILE = os.path.join(BASE_DIR, 'Datos_Crudos', 'dataset_integrador.csv')
OUTPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'dataset_gnn_granular_final.csv')

# 3. Creación automática de la carpeta de salida
carpeta_salida = os.path.dirname(OUTPUT_FILE)
os.makedirs(carpeta_salida, exist_ok=True)
print(f"Directorio de salida preparado: {carpeta_salida}")

# 4. Verificación de archivo de entrada
if os.path.exists(INPUT_FILE):
    print("¡Archivo de origen encontrado! Todo listo para iniciar.")
else:
    print(f"ERROR CRÍTICO: No se encontró el archivo en {INPUT_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de salida preparado: /content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN/Datos_Procesados
¡Archivo de origen encontrado! Todo listo para iniciar.


# PASO 3: Definición de Funciones del ETL (Fase 1 y 2)

In [8]:
# ==============================================================================
# CELDA 3: FUNCIONES DE FILTRADO Y TIEMPO
# ==============================================================================
def limpiar_y_filtrar(df):
    print("Fase 1: Filtrado Histórico y Categórico...")
    df = df[df['ANIO'] >= 2022].copy()

    # Validar existencia de columna DPTO_HECHO_NEW
    if 'DPTO_HECHO_NEW' in df.columns:
        df['DPTO_HECHO_NEW'] = df['DPTO_HECHO_NEW'].astype(str).str.strip().str.upper()
        df = df[df['DPTO_HECHO_NEW'].str.contains('LIMA', na=False)]
    else:
        print("Advertencia: No se encontró 'DPTO_HECHO_NEW', omitiendo filtro departamental.")

    # Conservar solo delitos predatorios
    modalidades_ok = 'ROBO|HURTO'
    modalidades_bad = 'ESTAFA|EXTORSIÓN|EXTORSION|CIBER'

    if 'P_MODALIDADES' in df.columns:
        df['P_MODALIDADES'] = df['P_MODALIDADES'].astype(str).str.strip().str.upper()
        df = df[df['P_MODALIDADES'].str.contains(modalidades_ok, na=False, regex=True)]
        df = df[~df['P_MODALIDADES'].str.contains(modalidades_bad, na=False, regex=True)]
    else:
        print("Advertencia: No se encontró 'P_MODALIDADES', omitiendo filtro de delitos.")

    return df

def generar_fechas_horas(anio, mes, cantidad):
    """Desagrega 'N' delitos de un mes en timestamps exactos con sesgo criminológico."""
    _, dias_en_mes = calendar.monthrange(anio, mes)
    fechas_generadas = []
    horas_generadas = []

    for _ in range(cantidad):
        # 1. Elegir día
        dia = random.choices(range(1, dias_en_mes + 1), k=1)[0]
        fecha_obj = datetime(anio, mes, dia)

        # Aumentar probabilidad si es fin de semana
        if fecha_obj.weekday() >= 4:
            if random.random() < 0.3:
               dia = random.choices(range(1, dias_en_mes + 1), k=1)[0]

        # 2. Elegir hora (Bimodal: 19:00 y 07:00)
        if random.random() < 0.7:
            hora = int(np.random.normal(19, 3))
        else:
            hora = int(np.random.normal(7, 2))

        hora = max(0, min(23, hora))
        minuto = random.randint(0, 59)

        fechas_generadas.append(f"{anio}-{mes:02d}-{dia:02d}")
        horas_generadas.append(f"{hora:02d}:{minuto:02d}:00")

    return fechas_generadas, horas_generadas

# PASO 4: Función de Imputación Espacial (OSMnx)

In [9]:
# ==============================================================================
# CELDA 4: FUNCIÓN ESPACIAL (OSMnx)
# ==============================================================================
def imputar_espacio_distrito(distrito, cantidad):
    """
    Descarga la red vial y asigna 'N' delitos con ruido gaussiano.
    """
    query = f"{distrito}, Lima, Peru"
    try:
        # Descargamos el grafo vial
        G = ox.graph_from_place(query, network_type='drive')
        gdf_nodes, _ = ox.graph_to_gdfs(G)

        if gdf_nodes.empty:
            return [], []

        lats = []
        lons = []

        nodos_muestreados = gdf_nodes.sample(n=cantidad, replace=True)

        for idx, row in nodos_muestreados.iterrows():
            # Ruido gaussiano (~15 metros)
            ruido_lat = np.random.normal(0, 0.00013)
            ruido_lon = np.random.normal(0, 0.00013)

            lats.append(row.geometry.y + ruido_lat)
            lons.append(row.geometry.x + ruido_lon)

        return lats, lons
    except Exception as e:
        print(f"  [!] Error en OSMnx para {distrito}: {e}")
        return [None]*cantidad, [None]*cantidad

# PASO 5: Orquestador y Ejecución Final

In [10]:
# ==============================================================================
# CELDA 5: EJECUCIÓN DEL PIPELINE
# ==============================================================================
def run_pipeline():
    print("Iniciando Pipeline ETL Espaciotemporal...")
    df_raw = pd.read_csv(INPUT_FILE, encoding='utf-8')
    df_clean = limpiar_y_filtrar(df_raw)

    # Agrupamos por mes y distrito
    agrupado = df_clean.groupby(['ANIO', 'MES', 'DIST_HECHO', 'P_MODALIDADES'])['cantidad'].sum().reset_index()

    registros_finales = []
    distritos_unicos = agrupado['DIST_HECHO'].unique()

    for distrito in distritos_unicos:
        print(f"\nProcesando topología de: {distrito}...")
        df_distrito = agrupado[agrupado['DIST_HECHO'] == distrito]
        total_delitos_distrito = int(df_distrito['cantidad'].sum())

        # Evitar procesar si la suma es 0
        if total_delitos_distrito == 0:
            continue

        lats_distrito, lons_distrito = imputar_espacio_distrito(distrito, total_delitos_distrito)

        if not lats_distrito:
            print(f"Saltando {distrito} por falla espacial.")
            continue

        puntero_coord = 0

        for _, row in df_distrito.iterrows():
            cant = int(row['cantidad'])
            if cant == 0: continue

            fechas, horas = generar_fechas_horas(row['ANIO'], row['MES'], cant)

            lats = lats_distrito[puntero_coord : puntero_coord + cant]
            lons = lons_distrito[puntero_coord : puntero_coord + cant]
            puntero_coord += cant

            for i in range(cant):
                # Validación extra para evitar errores de índice si OSMnx retornó menos nodos
                if i < len(lats) and lats[i] is not None:
                    registros_finales.append({
                        'fecha_delito': fechas[i],
                        'hora_delito': horas[i],
                        'distrito': distrito,
                        'tipo_delito': row['P_MODALIDADES'],
                        'latitud': lats[i],
                        'longitud': lons[i],
                        'es_dato_sintetico': True,
                        'metodo_imputacion': 'OSM_RoadNetwork_Gaussian'
                    })

    # Crear DataFrame granular final
    df_granular = pd.DataFrame(registros_finales)

    # Guardar resultados
    output_parquet = OUTPUT_FILE.replace('.csv', '.parquet')
    df_granular.to_parquet(output_parquet, index=False)
    df_granular.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

    print("\n=========================================")
    print(f"PIPELINE COMPLETADO. Total microdatos generados: {len(df_granular)}")
    print(f"Guardado en Parquet: {output_parquet}")
    print("=========================================")

# Ejecutar el proceso
run_pipeline()

Iniciando Pipeline ETL Espaciotemporal...
Fase 1: Filtrado Histórico y Categórico...

Procesando topología de: ANCON...

Procesando topología de: ASIA...

Procesando topología de: ATE...

Procesando topología de: AUCALLAMA...

Procesando topología de: BARRANCA...

Procesando topología de: BARRANCO...

Procesando topología de: BREÑA...

Procesando topología de: CALETA DE CARQUIN...

Procesando topología de: CARABAYLLO...

Procesando topología de: CATAHUASI...

Procesando topología de: CAUJUL...

Procesando topología de: CERRO AZUL...

Procesando topología de: CHACLACAYO...

Procesando topología de: CHANCAY...

Procesando topología de: CHILCA...

Procesando topología de: CHORRILLOS...

Procesando topología de: CIENEGUILLA...

Procesando topología de: COAYLLO...

Procesando topología de: COLONIA...

Procesando topología de: COMAS...

Procesando topología de: EL AGUSTINO...

Procesando topología de: GORGOR...

Procesando topología de: HUACHO...

Procesando topología de: HUALMAY...

Procesa